# Superpixels: segmenting HEALPix maps with SLIC

Superpixels oversegment a map into a few hundred small regions that group
neighbouring pixels with similar values: instead of working with millions of
individual pixels, a downstream classifier or statistic can work with a few
hundred coherent regions that mostly respect the edges of the map.
`healpy.superpixels` implements the SLIC algorithm (Achanta et al. 2012) on
the sphere: the spatial distance between a pixel and a cluster centre is the
angular distance, and the clustering runs directly on the HEALPix grid.

This tutorial shows how to segment a map, how to tune the result with the
`compactness` parameter, and how to use the segmentation downstream. See the
[API documentation](https://healpy.readthedocs.io/en/latest/generated/healpy.superpixels.html)
for the full parameter reference.

In [ ]:
import healpy as hp
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (10, 5)

## A test map

We use a synthetic map in Kelvin with a large-scale gradient and a few
Gaussian sources, so the superpixels have strong edges to snap to.

In [ ]:
nside = 64
npix = hp.nside2npix(nside)
theta, phi = hp.pix2ang(nside, np.arange(npix))
vec = np.asarray(hp.pix2vec(nside, np.arange(npix)))

m = 4 * np.cos(theta)
for amp, center, sigma in [
    (3.0, hp.ang2vec(1.0, 0.5), np.radians(25)),
    (-2.0, hp.ang2vec(2.2, 3.5), np.radians(18)),
    (2.5, hp.ang2vec(1.4, 5.0), np.radians(14)),
]:
    angular_distance = np.arccos(np.clip(vec.T @ center, -1, 1))
    m += amp * np.exp(-0.5 * (angular_distance / sigma) ** 2)

hp.projview(m, title="Input map", unit="K", cmap="RdYlBu_r");

## First segmentation

`superpixels` returns one label per pixel. Every valid pixel gets a label in
`[0, n_segments)`; here the map has no missing pixels, so all of them are
labelled. Invalid pixels (`UNSEEN`, `NaN` or masked) would be labelled `-1`.

With `return_centers=True` we also get the centre pixel and the mean map
value of each superpixel, in the units of the input map. Reproducing the
input map with one value per superpixel is a good way to inspect what the
segmentation captured:

In [ ]:
n_segments = 64
labels, center_pix, center_values = hp.superpixels(
    m, n_segments, compactness=0.3, return_centers=True
)
m_averaged = center_values[labels, 0]

print(f"{n_segments} superpixels, {np.unique(labels).size} of them non-empty")

In [ ]:
def label_cmap(k, shuffle_seed=654321):
    # a qualitative colormap with a different color for each superpixel
    rng = np.random.default_rng(shuffle_seed)
    colors = plt.get_cmap("turbo")(np.linspace(0, 1, k))
    return matplotlib.colors.ListedColormap(colors[rng.permutation(k)])


def boundary_norm(k):
    # map each integer label to exactly one color of a k-color colormap
    return matplotlib.colors.BoundaryNorm(np.arange(-0.5, k + 0.5, 1), k)


def as_plot_map(labels):
    # labels as a float map, with invalid pixels shown as UNSEEN (grey)
    return np.where(labels >= 0, labels, hp.UNSEEN).astype(float)

In [ ]:
fig = plt.figure(figsize=(15, 4))
hp.projview(
    as_plot_map(labels), sub=131, title="Superpixel labels",
    cmap=label_cmap(n_segments), norm=boundary_norm(n_segments),
)
hp.projview(m, sub=132, title="Input map", unit="K", cmap="RdYlBu_r",
            min=np.floor(m.min()), max=np.ceil(m.max()))
hp.projview(
    m_averaged, sub=133, title="One value per superpixel", unit="K",
    cmap="RdYlBu_r", min=np.floor(m.min()), max=np.ceil(m.max()),
);

## `compactness`: how closely to follow the map

`compactness` balances the feature distance (difference in map value)
against the spatial distance (angle on the sky). Features are normalised to
`[0, 1]` internally, so `compactness` is dimensionless and its useful range
is between 0 and 1:

* close to 0: superpixels stretch along the contours of the map and the
  superpixel-averaged map is nearly identical to the input;
* around a few tenths: roughly round superpixels that still snap to strong
  edges;
* of order 1 and above: compact, regular superpixels, saturating to an
  almost purely spatial partition (a fancier `ud_grade`).

The rows below share one linear color scale, computed from the input map:

In [ ]:
compactness_values = [0.01, 0.3, 1.0]
vmin = np.floor(m.min())
vmax = np.ceil(m.max())
print(f"shared color scale: {vmin:.0f} to {vmax:.0f} K")

labels_sweep = {
    c: hp.superpixels(m, n_segments, compactness=c) for c in compactness_values
}

In [ ]:
fig = plt.figure(figsize=(15, 4))
for i, c in enumerate(compactness_values):
    hp.projview(
        as_plot_map(labels_sweep[c]), sub=(1, 3, i + 1),
        title=f"labels, compactness={c}",
        cmap=label_cmap(n_segments), norm=boundary_norm(n_segments),
    )

In [ ]:
fig = plt.figure(figsize=(15, 4))
hp.projview(
    m, sub=141, title="input map", unit="K", cmap="RdYlBu_r", min=vmin, max=vmax,
)
for i, c in enumerate(compactness_values):
    averaged = np.zeros(npix)
    for lbl in range(n_segments):
        averaged[labels_sweep[c] == lbl] = m[labels_sweep[c] == lbl].mean()
    hp.projview(
        averaged, sub=(1, 4, i + 2),
        title=f"averaged, compactness={c}", unit="K", cmap="RdYlBu_r",
        min=vmin, max=vmax,
    )

## How many superpixels?

`n_segments` sets the target number of regions; the runtime stays of order
seconds even for a few hundred segments at moderate `nside`:

In [ ]:
fig = plt.figure(figsize=(15, 4))
for i, k in enumerate([16, 64, 256]):
    labels_k = hp.superpixels(m, k, compactness=0.3)
    hp.projview(
        as_plot_map(labels_k), sub=(1, 3, i + 1), title=f"{k} superpixels",
        cmap=label_cmap(k), norm=boundary_norm(k),
    )

## Initialisation strategies

The initial cluster centres can be placed with three strategies:

* `init="hierarchical"` (the default): deterministic, seeds at the centres
  of the coarser HEALPix level closest to `n_segments`;
* `init="farthest"`: deterministic farthest-point sampling;
* `init="greedy"`: k-means++ seeding, random, so pass a `seed` to make it
  reproducible.

The three converge to very similar partitions:

In [ ]:
fig = plt.figure(figsize=(15, 4))
for i, (init, kwargs) in enumerate(
    [("hierarchical", {}), ("farthest", {}), ("greedy", {"seed": 12345})]
):
    labels_init = hp.superpixels(m, n_segments, compactness=0.3, init=init, **kwargs)
    hp.projview(
        as_plot_map(labels_init), sub=(1, 3, i + 1), title=f"init='{init}'",
        cmap=label_cmap(n_segments), norm=boundary_norm(n_segments),
    )

## Masked maps

Pixels that are `UNSEEN`, `NaN` or masked are never assigned to a
superpixel and keep the label `-1`. The spacing of the centres is
rescaled by the valid fraction of the sky, so a masked map keeps the same
superpixel density over the visible region, and every valid pixel gets a
label even for elongated masks:

In [ ]:
mask = np.abs(np.pi / 2 - theta) < np.radians(20)  # cut a band around the equator
m_masked = np.where(mask, hp.UNSEEN, m)
labels_masked, _, center_values_masked = hp.superpixels(
    m_masked, 32, compactness=0.3, return_centers=True
)
print(
    f"{(labels_masked == -1).sum()} masked pixels, "
    f"{(labels_masked >= 0).sum()} valid pixels all labelled: "
    f"{bool(np.all(labels_masked[~mask] >= 0))}"
)

fig = plt.figure(figsize=(10, 4))
hp.projview(
    as_plot_map(labels_masked), sub=121, title="Labels, masked map",
    cmap=label_cmap(32), norm=boundary_norm(32),
)
hp.projview(
    m_masked, sub=122, title="Masked input map", unit="K", cmap="RdYlBu_r",
);

## Multi-feature maps

A stack of maps with the same pixelisation is clustered in the combined
feature space. Each feature is normalised to `[0, 1]` independently, so the
features may have completely different units and scales and still
contribute equally:

In [ ]:
m2 = 2 * np.sin(theta) * np.cos(2 * phi)

labels_multi = hp.superpixels(np.stack([m, m2]), n_segments, compactness=0.3)
labels_m1 = hp.superpixels(m, n_segments, compactness=0.3)

fig = plt.figure(figsize=(15, 4))
hp.projview(
    m2, sub=131, title="second feature map", unit="K", cmap="RdYlBu_r",
)
hp.projview(
    as_plot_map(labels_m1), sub=132, title="labels, first feature only",
    cmap=label_cmap(n_segments), norm=boundary_norm(n_segments),
)
hp.projview(
    as_plot_map(labels_multi), sub=133, title="labels, both features",
    cmap=label_cmap(n_segments), norm=boundary_norm(n_segments),
);

## Using the segmentation downstream

The point of superpixels is to replace millions of pixels with a few
hundred regions. `center_values` already gives the mean of each feature per
superpixel (in the units of the input map); together with the number of
pixels per superpixel you have a compact per-region summary:

In [ ]:
counts = np.bincount(labels[labels >= 0], minlength=n_segments)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(center_values[:, 0], bins=16, color="C0")
axes[0].set_xlabel("superpixel mean [K]")
axes[0].set_ylabel("superpixels")
axes[1].hist(counts, bins=16, color="C1")
axes[1].set_xlabel("superpixel size [pixels]")
axes[1].set_ylabel("superpixels");

## Caveats

Two properties are inherent to SLIC (the same holds for 2D images) and
worth knowing before using the segmentation:

* **Not resolution-invariant**: the same sky observed at several `nside`
  with the same `n_segments` gives partitions of the same quality but
  spatially different ones (adjusted Rand index of roughly 0.5 to 0.7
  between adjacent resolutions). Use superpixels for region statistics at
  one resolution, not to track structures across resolutions.
* **Not guaranteed connected**: in particular on noisy maps at low
  `compactness`, a superpixel can split into several patches. If you need
  connected regions, merge the small patches into their neighbours or look
  at SNIC (Achanta & Süsstrunk 2017).

Extensive testing of this implementation, including the resolution and
connectivity discussion, is described in
[this blog post](https://www.zonca.dev/posts/2026-10-09-slic-superpixels-on-the-healpix-grid).